In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
import yaml

# Project paths (the notebook lives in notebooks/)
ROOT = Path("..")
RAW_OSM = ROOT / "data" / "raw" / "osm"
INTERIM = ROOT / "data" / "interim"
RAW_OSM.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

with open(ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

COUNTRIES = config["scope"]["countries"]
PAUSE_S = config["osm"]["request_pause_s"]
print(len(COUNTRIES), "countries | pause:", PAUSE_S, "s")

30 countries | pause: 5 s


In [2]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def build_query(country_code: str) -> str:
    """Overpass QL query for data centres in one country (existing, under construction, planned)."""
    return f"""
    [out:json][timeout:180];
    area["ISO3166-1"="{country_code}"][admin_level=2]->.country;
    (
      nwr["telecom"="data_center"](area.country);
      nwr["building"="data_center"](area.country);
      nwr["industrial"="data_centre"](area.country);
      nwr["construction:telecom"="data_center"](area.country);
      nwr["construction"="data_center"](area.country);
      nwr["proposed:telecom"="data_center"](area.country);
    );
    out geom;
    """

print(build_query("PT"))


    [out:json][timeout:180];
    area["ISO3166-1"="PT"][admin_level=2]->.country;
    (
      nwr["telecom"="data_center"](area.country);
      nwr["building"="data_center"](area.country);
      nwr["industrial"="data_centre"](area.country);
      nwr["construction:telecom"="data_center"](area.country);
      nwr["construction"="data_center"](area.country);
      nwr["proposed:telecom"="data_center"](area.country);
    );
    out geom;
    


In [4]:
HEADERS = {"User-Agent": "ia-agua-datacenters (portfolio project; github.com/Carol-Sust/Sustainability)"}

def fetch_country(country_code: str, force: bool = False, max_retries: int = 4) -> dict:
    """Download OSM data centres for one country, caching the raw JSON in data/raw/osm/."""
    cache_file = RAW_OSM / f"{country_code}.json"
    if cache_file.exists() and not force:
        return json.loads(cache_file.read_text())

    query = build_query(country_code)
    for attempt in range(1, max_retries + 1):
        response = requests.post(OVERPASS_URL, data={"data": query}, headers=HEADERS, timeout=300)
        if response.status_code == 200:
            data = response.json()
            cache_file.write_text(json.dumps(data))
            return data
        # 429 = too many requests, 504 = server busy -> wait and retry
        wait = 30 * attempt
        print(f"{country_code}: HTTP {response.status_code}, retry {attempt}/{max_retries} in {wait}s")
        time.sleep(wait)

    raise RuntimeError(f"{country_code}: failed after {max_retries} attempts")

In [5]:
pt = fetch_country("PT")
elements = pt["elements"]
print("Elements found:", len(elements))

# How many points (node), polygons (way) and relations?
print(pd.Series([e["type"] for e in elements]).value_counts())

# Tags of each element as a table
tags = pd.json_normalize([e.get("tags", {}) for e in elements])
cols = [c for c in ["name", "operator", "telecom", "building", "construction", "proposed:telecom"] if c in tags.columns]
tags[cols].head(20)

PT: HTTP 502, retry 1/4 in 30s
Elements found: 19
way     15
node     4
Name: count, dtype: int64


,name,operator,telecom,building
0,NaN,MEO,data_center,NaN
1,Centro de Computação Avançada do Minho,Fundação para a Ciência e Tecnologia,data_center,NaN
2,NaN,NaN,data_center,NaN
3,NaN,NaN,data_center,NaN
4,Equinix LS1,Equinix,data_center,data_center
5,NOS Imopolis II,NOS,data_center,data_center
6,NOS Imopolis I,NOS,data_center,data_center
7,Seixal Colocation Site,TATA Communications,data_center,yes
8,Data Center Portugal Telecom da Covilhã,Portugal Telecom,data_center,yes
9,Datacenter de Riba d'Ave,REN;NOS,data_center,data_center


In [6]:
from tqdm import tqdm

counts = {}
failed = []

for cc in tqdm(COUNTRIES):
    already_cached = (RAW_OSM / f"{cc}.json").exists()
    try:
        data = fetch_country(cc)
        counts[cc] = len(data["elements"])
    except RuntimeError as error:
        print(error)
        failed.append(cc)
    # Only pause after a real request, not when reading from cache
    if not already_cached:
        time.sleep(PAUSE_S)

print("Failed:", failed if failed else "none")

  3%|█▍                                         | 1/30 [00:38<18:42, 38.69s/it]

BE: HTTP 502, retry 1/4 in 30s
BE: HTTP 502, retry 2/4 in 60s


 13%|█████▋                                     | 4/30 [04:27<26:16, 60.65s/it]

CY: HTTP 502, retry 1/4 in 30s


 27%|███████████▍                               | 8/30 [07:44<17:59, 49.06s/it]

FI: HTTP 502, retry 1/4 in 30s
FI: HTTP 504, retry 2/4 in 60s
FI: HTTP 502, retry 3/4 in 90s


... (14 outputs left out)

 97%|████████████████████████████████████████▌ | 29/30 [35:15<00:43, 43.97s/it]

CH: HTTP 502, retry 1/4 in 30s


100%|██████████████████████████████████████████| 30/30 [36:32<00:00, 73.08s/it]

Failed: ['DE']
